<a href="https://colab.research.google.com/github/zinhcandoit/ViHandwrittenOCR/blob/ver1/Deepseek_OCR_(3B).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Installation

In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install jiwer
!pip install einops addict easydict

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Copy file từ Drive sang bộ nhớ tạm của Colab để giải nén cho nhanh
!cp "/content/drive/MyDrive/data.zip" /content/data.zip
!unzip /content/data.zip

In [ ]:
# Have to install pytorch with cuda support first
# !pip uninstall torch torchvision torchaudio
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

In [ ]:

# !pip install --upgrade --force-reinstall --no-cache-dir unsloth transformers

### Import Library

In [ ]:
from unsloth import FastVisionModel # FastLanguageModel for LLMs
import torch
from transformers import AutoModel
from huggingface_hub import snapshot_download
import random
import os
import json
import cv2
import numpy as np
from PIL import Image
from tqdm import tqdm # Thanh hiển thị tiến độ

os.environ["UNSLOTH_WARN_UNINITIALIZED"] = '0'


### Unsloth

Let's prepare the OCR model to our local first

In [ ]:
snapshot_download("unsloth/DeepSeek-OCR", local_dir = "deepseek_ocr")

In [ ]:
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Phiên bản Torch: {torch.__version__}")
if torch.cuda.is_available():
    print(f"Current device: {torch.cuda.current_device()}")
    print(f"Device name: {torch.cuda.get_device_name(0)}")

In [ ]:
# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Qwen3-VL-8B-Instruct-bnb-4bit", # Qwen 3 vision support
    "unsloth/Qwen3-VL-8B-Thinking-bnb-4bit",
    "unsloth/Qwen3-VL-32B-Instruct-bnb-4bit",
    "unsloth/Qwen3-VL-32B-Thinking-bnb-4bit",
] # More models at https://huggingface.co/unsloth

### HELPER

In [ ]:
# --- CẤU HÌNH TIỀN XỬ LÝ ---
TARGET_HEIGHT = 192  # Yêu cầu resize về 192px
INSTRUCTION = "<image>\nConvert the following image of Vietnamese writing into digital t" \
"ext, keeping the tone diacritics and vowel diacritics in Vietnamese text."

To format the dataset, all vision finetuning tasks should be formatted as follows:

```python
[
{ "role": "<|User|>",
  "content": "",
  "images": []
},
{ "role": "<|Assistant|>",
  "content": ""
},
]
```

In [ ]:
def crawl_and_prepare_data(
    root_path,
    seed=42,
    select_count=None  # None = lấy hết
):
    # 1. Lấy danh sách thư mục THỰC
    dir_names = [
        d for d in os.listdir(root_path)
        if os.path.isdir(os.path.join(root_path, d))
    ]

    if len(dir_names) == 0:
        raise RuntimeError(f"No subdirectories found in {root_path}")

    # 2. Shuffle để random hóa
    random.seed(seed)
    random.shuffle(dir_names)

    # 3. Chọn N thư mục nếu cần
    if select_count is not None:
        dir_names = dir_names[:select_count]

    print(f"--> Chọn {len(dir_names)} thư mục từ {root_path}")

    raw_dataset = []

    for dir_name in dir_names:
        dir_path = os.path.join(root_path, dir_name)
        json_path = os.path.join(dir_path, "label.json")

        if not os.path.exists(json_path):
            print(f"⚠️  Không tìm thấy label.json trong {dir_path}")
            continue

        with open(json_path, "r", encoding="utf-8") as f:
            labels = json.load(f)

        for img_name, gt in labels.items():
            img_path = os.path.join(dir_path, img_name)
            if os.path.exists(img_path):
                raw_dataset.append({
                    "image_path": img_path,
                    "text": gt
                })

    return raw_dataset

def convert_to_unsloth_format(pil_img, text):
    """Format dữ liệu chuẩn cho Unsloth SFTTrainer"""
    return {
        "messages": [
            {
                "role": "<|User|>",
                "content": INSTRUCTION,
                "images": [pil_img] # Unsloth yêu cầu list ảnh
            },
            {
                "role": "<|Assistant|>",
                "content": text
            }
        ]
    }

### Let's Evaluate Deepseek-OCR Baseline Performance

In [ ]:
model, tokenizer = FastVisionModel.from_pretrained(
    "./deepseek_ocr",
    load_in_4bit = False, # Use 4bit to reduce memory use. False for 16bit LoRA.
    auto_model = AutoModel,
    trust_remote_code=True,
    unsloth_force_compile=True,
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for long context

)

You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR. This is not supported for all configurations of models and can yield errors.


Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2025.12.9: Fast Deepseekocr patching. Transformers: 4.56.2.
   \\   /|    NVIDIA GeForce RTX 3070 Laptop GPU. Num GPUs = 1. Max memory: 8.0 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.9.1+cu126. CUDA: 8.6. CUDA Toolkit: 12.6. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR. This is not supported for all configurations of models and can yield errors.
You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR. This is not supported for all configurations of models and can yield errors.


Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Some weights of DeepseekOCRForCausalLM were not initialized from the model checkpoint at ./deepseek_ocr and are newly initialized: ['model.vision_model.embeddings.position_ids']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# prompt = "<image>\nFree OCR. "
prompt = "<image>\nFree OCR. "
image_file = 'data/UIT_HWDB_line/train_data/4/1.jpg'
output_path = 'output'
# infer(self, tokenizer, prompt='', image_file='', output_path = ' ', base_size = 1024, image_size = 640, crop_mode = True, test_compress = False, save_results = False):

# Tiny: base_size = 512, image_size = 512, crop_mode = False
# Small: base_size = 640, image_size = 640, crop_mode = False
# Base: base_size = 1024, image_size = 1024, crop_mode = False
# Large: base_size = 1280, image_size = 1280, crop_mode = False

# Gundam: base_size = 1024, image_size = 640, crop_mode = True

res = model.infer(tokenizer, prompt=prompt, image_file=image_file, output_path = output_path, base_size = 1024, image_size = 640, crop_mode=True, save_results = True, test_compress = False)


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([9, 100, 1280])
Fógo y di tíhoá chái tíoh huagá diu thí hánh Luát tát tái. Hán ché-
===============save results:===============


image: 0it [00:00, ?it/s]
other: 0it [00:00, ?it/s]


<h3>Baseline Model Performance: 23% Character Error Rate (CER) for this sample !</h3>

# Let's finetune Deepseek-OCR !

We now add LoRA adapters for parameter efficient finetuning - this allows us to only efficiently train 1% of all parameters.

**[NEW]** We also support finetuning ONLY the vision part of the model, or ONLY the language part. Or you can select both! You can also select to finetune the attention or the MLP layers!

In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],

    r = 16,           # The larger, the higher the accuracy, but might overfit
    lora_alpha = 16,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
    # target_modules = "all-linear", # Optional now! Can specify a list if needed
)

Data crawling

In [ ]:
DATA_ROOT = "data/UIT_HWDB_line/train_data"
NUM_DIRS_TO_SELECT = 38  # ~ 1000 data

raw_data = crawl_and_prepare_data(DATA_ROOT, 42, NUM_DIRS_TO_SELECT)

print(f"\nTổng số mẫu (ảnh + text) thu thập được: {len(raw_data)}")
print("Ví dụ 3 mẫu đầu tiên:")
for sample in raw_data[:3]:
    print(f"Ảnh: {sample['image_path']}")
    print(f"Text: {sample['text']}")
    print("-" * 20)

Image preprocessing

In [ ]:
import cv2
import numpy as np
import os
import random
from tqdm import tqdm
from scipy.ndimage import gaussian_filter, map_coordinates # Cần thiết cho Elastic Transform

# --- CẤU HÌNH ---
BACKGROUND_ROOT_DIR = "data/background_img" # Trong này cần có folder: low, medium, high
OUTPUT_ROOT_DIR = "data/train_dataset"
os.makedirs(OUTPUT_ROOT_DIR, exist_ok=True)

# --- CÁC HÀM XỬ LÝ ẢNH NÂNG CAO ---

def elastic_transform(image, alpha=30, sigma=8, random_state=None):
    """
    Biến dạng đàn hồi (Elastic Transform) trên ảnh Grayscale.
    """
    if random_state is None:
        random_state = np.random.RandomState(None)

    shape = image.shape
    dx = gaussian_filter((random_state.rand(*shape) * 2 - 1), sigma) * alpha
    dy = gaussian_filter((random_state.rand(*shape) * 2 - 1), sigma) * alpha

    x, y = np.meshgrid(np.arange(shape[1]), np.arange(shape[0]))
    indices = np.reshape(y+dy, (-1, 1)), np.reshape(x+dx, (-1, 1))

    # mode='reflect' giúp lấp đầy các khoảng trống ở biên
    distorted_image = map_coordinates(image, indices, order=1, mode='reflect').reshape(shape)
    return distorted_image

def calculate_stroke_density(gray_img):
    """Tính mật độ nét chữ để quyết định đậm/mảnh"""
    # Otsu tạm thời để tính toán
    _, binary = cv2.threshold(gray_img, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    stroke_pixels = np.count_nonzero(binary)
    total_pixels = binary.shape[0] * binary.shape[1]
    return (stroke_pixels / total_pixels) * 100

def modify_thickness(gray_img, mode='thin'):
    """
    Thay đổi độ dày nét chữ trên ảnh Grayscale.
    - Text đen, nền trắng.
    - Dilate (Cực đại): Vùng sáng mở rộng -> Chữ bị ăn mòn -> Mảnh đi (Thin).
    - Erode (Cực tiểu): Vùng tối mở rộng -> Chữ dày lên (Thick).
    """
    kernel = np.ones((2, 2), np.uint8) # Kernel nhỏ để thay đổi nhẹ 2%
    if mode == 'thin':
        return cv2.dilate(gray_img, kernel, iterations=1)
    elif mode == 'thick':
        return cv2.erode(gray_img, kernel, iterations=1)
    return gray_img

def clean_image_otsu(img):
    """Làm sạch ảnh sang Binary"""
    if len(img.shape) == 3:
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    else:
        gray = img
    # Chuyển sang ảnh nhị phân (chữ đen, nền trắng chuẩn)
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return binary

def apply_paper_texture(fg_binary, bg_path):
    """Ghép nền giấy"""
    bg = cv2.imread(bg_path)
    if bg is None: return None

    h_fg, w_fg = fg_binary.shape[:2]
    h_bg, w_bg = bg.shape[:2]

    # Resize ảnh nền nếu nhỏ hơn ảnh chữ
    if h_bg < h_fg or w_bg < w_fg:
        scale = max(h_fg/h_bg, w_fg/w_bg) * 1.5
        bg = cv2.resize(bg, None, fx=scale, fy=scale)
        h_bg, w_bg = bg.shape[:2]

    # Crop ngẫu nhiên
    x = random.randint(0, w_bg - w_fg)
    y = random.randint(0, h_bg - h_fg)
    bg_crop = bg[y:y+h_fg, x:x+w_fg]

    # Hòa trộn
    fg_bgr = cv2.cvtColor(fg_binary, cv2.COLOR_GRAY2BGR)
    fg_float = fg_bgr.astype(float)
    bg_float = bg_crop.astype(float)
    blended = (fg_float * bg_float) / 255.0

    return np.clip(blended, 0, 255).astype(np.uint8)

# --- HÀM QUẢN LÝ BACKGROUND ---

def load_background_library(root_dir):
    """
    Quét và phân loại background vào dictionary
    """
    library = {'low': [], 'medium': [], 'high': []}
    valid_exts = {".jpg", ".jpeg", ".png", ".bmp"}

    for category in library.keys():
        folder_path = os.path.join(root_dir, category)
        if os.path.exists(folder_path):
            for f in os.listdir(folder_path):
                if os.path.splitext(f)[1].lower() in valid_exts:
                    library[category].append(os.path.join(folder_path, f))

    total_bg = sum(len(v) for v in library.values())
    return library, total_bg

# --- PIPELINE CHÍNH ---

def process_and_rebuild_structure(raw_dataset, bg_root, output_dir):
    new_dataset_structure = []

    # 1. Load Backgrounds
    bg_library, total_bg_count = load_background_library(bg_root)
    print(f"--> Tìm thấy tổng cộng {total_bg_count} ảnh background.")
    print(f"--> Chi tiết: Low: {len(bg_library['low'])}, Medium: {len(bg_library['medium'])}, High: {len(bg_library['high'])}")

    # Counter cho tên file (0001.jpg -> xxxx.jpg)
    global_counter = 1

    # Duyệt qua từng ảnh gốc (Input Raw)
    for item in tqdm(raw_dataset):
        original_path = item['image_path']
        label_text = item['text']

        img_origin_bgr = cv2.imread(original_path)
        if img_origin_bgr is None: continue

        # Chuyển sang Grayscale ngay từ đầu để xử lý Augment
        img_origin_gray = cv2.cvtColor(img_origin_bgr, cv2.COLOR_BGR2GRAY)

        # ====================================================
        # PHẦN 1: LƯU ẢNH GỐC (Đã Clean Otsu nhưng ko ghép nền)
        # ====================================================
        # Vẫn lưu 1 bản sạch không nền làm base
        img_clean_base = clean_image_otsu(img_origin_gray)
        filename_base = f"{global_counter:04d}.jpg"
        save_path_base = os.path.join(output_dir, filename_base)
        cv2.imwrite(save_path_base, img_clean_base)

        new_dataset_structure.append({"image_path": save_path_base, "text": label_text})
        global_counter += 1

        # ====================================================
        # PHẦN 2: GHÉP BACKGROUND THEO LOGIC RIÊNG
        # ====================================================

        # Duyệt qua từng loại background (low, medium, high)
        for category, bg_list in bg_library.items():
            for bg_path in bg_list:

                # --- LOGIC XỬ LÝ ẢNH TRƯỚC KHI CLEAN ---
                img_to_process = img_origin_gray.copy()

                # Logic 1: Nền Low -> Elastic Transform
                if category == 'low':
                    img_to_process = elastic_transform(img_to_process, alpha=10, sigma=3)

                # Logic 2: Nền Medium -> Check mật độ -> Dày/Mảnh
                elif category == 'medium':
                    density = calculate_stroke_density(img_to_process)
                    if 6 <= density <= 8:
                        # Mảnh hơn (Dilation trên nền trắng)
                        img_to_process = modify_thickness(img_to_process, mode='thin')
                    elif 8 < density <= 10:
                        # Dày hơn (Erosion trên nền trắng)
                        img_to_process = modify_thickness(img_to_process, mode='thick')
                    # Ngoài khoảng này thì giữ nguyên

                # Logic 3: Nền High -> Giữ nguyên (Không làm gì cả)

                # --- SAU KHI BIẾN DẠNG MỚI ĐƯA VÀO CLEAN ---
                img_cleaned = clean_image_otsu(img_to_process)

                # --- GHÉP NỀN ---
                img_final = apply_paper_texture(img_cleaned, bg_path)

                if img_final is not None:
                    filename_aug = f"{global_counter:04d}.jpg"
                    save_path_aug = os.path.join(output_dir, filename_aug)

                    cv2.imwrite(save_path_aug, img_final)

                    new_dataset_structure.append({
                        "image_path": save_path_aug,
                        "text": label_text
                    })
                    global_counter += 1

    return new_dataset_structure

# --- CHẠY THỬ ---
# raw_data là list dictionary [{'image_path': '...', 'text': '...'}] đã có từ trước
pre_dataset = process_and_rebuild_structure(
    raw_dataset=raw_data,
    bg_root=BACKGROUND_ROOT_DIR,
    output_dir=OUTPUT_ROOT_DIR
)
print(f"Hoàn tất! Tổng số ảnh trong dataset mới: {len(pre_dataset)}")

In [ ]:


# --- THỰC THI PIPELINE ---
final_dataset = []

print(f"Đang xử lý {len(pre_dataset)} mẫu dữ liệu...")

# Sử dụng tqdm để hiện thanh loading
for sample in tqdm(pre_dataset):
    processed_cv2_img = cv2.imread(sample['image_path'])
    if sample is not None:
        processed_cv2_img = cv2.cvtColor(processed_cv2_img, cv2.COLOR_BGR2RGB)
        # B. Chuyển sang PIL Image
        pil_img = Image.fromarray(processed_cv2_img)
        # C. Đóng gói vào format Conversation
        formatted_sample = convert_to_unsloth_format(pil_img, sample['text'])
        final_dataset.append(formatted_sample)

print(f"\nHoàn tất! Số lượng mẫu hợp lệ: {len(final_dataset)}")

We look at how the conversations are structured for the first example:

In [ ]:

import matplotlib.pyplot as plt

print("\n--- KIỂM TRA MẪU ĐẦU TIÊN SAU KHI XỬ LÝ ---")
check_sample = final_dataset[30]['messages']
img_check = check_sample[0]['images'][0]
text_check = check_sample[1]['content']

print(f"Kích thước ảnh: {img_check.size} (W, H)") # H phải là 192
print(f"Ground Truth: {text_check}")

plt.figure(figsize=(10, 4))
plt.imshow(img_check)
plt.axis('off')
plt.title("Ảnh đã Resize (H=192) & Tiền xử lý")
plt.show()

In [ ]:
# @title Create datacollator

import torch
import math
from dataclasses import dataclass
from typing import Dict, List, Any, Tuple
from PIL import Image, ImageOps
from torch.nn.utils.rnn import pad_sequence
import io

from deepseek_ocr.modeling_deepseekocr import (
    format_messages,
    text_encode,
    BasicImageTransform,
    dynamic_preprocess,
)

@dataclass
class DeepSeekOCRDataCollator:
    """
    Args:
        tokenizer: Tokenizer
        model: Model
        image_size: Size for image patches (default: 640)
        base_size: Size for global view (default: 1024)
        crop_mode: Whether to use dynamic cropping for large images
        train_on_responses_only: If True, only train on assistant responses (mask user prompts)
    """
    tokenizer: Any
    model: Any
    image_size: int = 640
    base_size: int = 1024
    crop_mode: bool = True
    image_token_id: int = 128815
    train_on_responses_only: bool = True

    def __init__(
        self,
        tokenizer,
        model,
        image_size: int = 640,
        base_size: int = 1024,
        crop_mode: bool = True,
        train_on_responses_only: bool = True,
    ):
        self.tokenizer = tokenizer
        self.model = model
        self.image_size = image_size
        self.base_size = base_size
        self.crop_mode = crop_mode
        self.image_token_id = 128815
        self.dtype = model.dtype  # Get dtype from model
        self.train_on_responses_only = train_on_responses_only

        self.image_transform = BasicImageTransform(
            mean=(0.5, 0.5, 0.5),
            std=(0.5, 0.5, 0.5),
            normalize=True
        )
        self.patch_size = 16
        self.downsample_ratio = 4

        # Get BOS token ID from tokenizer
        if hasattr(tokenizer, 'bos_token_id') and tokenizer.bos_token_id is not None:
            self.bos_id = tokenizer.bos_token_id
        else:
            self.bos_id = 0
            print(f"Warning: tokenizer has no bos_token_id, using default: {self.bos_id}")

    def deserialize_image(self, image_data) -> Image.Image:
        """Convert image data (bytes dict or PIL Image) to PIL Image in RGB mode"""
        if isinstance(image_data, Image.Image):
            return image_data.convert("RGB")
        elif isinstance(image_data, dict) and 'bytes' in image_data:
            image_bytes = image_data['bytes']
            image = Image.open(io.BytesIO(image_bytes))
            return image.convert("RGB")
        else:
            raise ValueError(f"Unsupported image format: {type(image_data)}")

    def calculate_image_token_count(self, image: Image.Image, crop_ratio: Tuple[int, int]) -> int:
        """Calculate the number of tokens this image will generate"""
        num_queries = math.ceil((self.image_size // self.patch_size) / self.downsample_ratio)
        num_queries_base = math.ceil((self.base_size // self.patch_size) / self.downsample_ratio)

        width_crop_num, height_crop_num = crop_ratio

        if self.crop_mode:
            img_tokens = num_queries_base * num_queries_base + 1
            if width_crop_num > 1 or height_crop_num > 1:
                img_tokens += (num_queries * width_crop_num + 1) * (num_queries * height_crop_num)
        else:
            img_tokens = num_queries * num_queries + 1

        return img_tokens

    def process_image(self, image: Image.Image) -> Tuple[List, List, List, List, Tuple[int, int]]:
        """
        Process a single image based on crop_mode and size thresholds

        Returns:
            Tuple of (images_list, images_crop_list, images_spatial_crop, tokenized_image, crop_ratio)
        """
        images_list = []
        images_crop_list = []
        images_spatial_crop = []

        if self.crop_mode:
            # Determine crop ratio based on image size
            if image.size[0] <= 640 and image.size[1] <= 640:
                crop_ratio = (1, 1)
                images_crop_raw = []
            else:
                images_crop_raw, crop_ratio = dynamic_preprocess(
                    image, min_num=2, max_num=9,
                    image_size=self.image_size, use_thumbnail=False
                )

            # Process global view with padding
            global_view = ImageOps.pad(
                image, (self.base_size, self.base_size),
                color=tuple(int(x * 255) for x in self.image_transform.mean)
            )
            images_list.append(self.image_transform(global_view).to(self.dtype))

            width_crop_num, height_crop_num = crop_ratio
            images_spatial_crop.append([width_crop_num, height_crop_num])

            # Process local views (crops) if applicable
            if width_crop_num > 1 or height_crop_num > 1:
                for crop_img in images_crop_raw:
                    images_crop_list.append(
                        self.image_transform(crop_img).to(self.dtype)
                    )

            # Calculate image tokens
            num_queries = math.ceil((self.image_size // self.patch_size) / self.downsample_ratio)
            num_queries_base = math.ceil((self.base_size // self.patch_size) / self.downsample_ratio)

            tokenized_image = ([self.image_token_id] * num_queries_base + [self.image_token_id]) * num_queries_base
            tokenized_image += [self.image_token_id]

            if width_crop_num > 1 or height_crop_num > 1:
                tokenized_image += ([self.image_token_id] * (num_queries * width_crop_num) + [self.image_token_id]) * (
                    num_queries * height_crop_num)

        else:  # crop_mode = False
            crop_ratio = (1, 1)
            images_spatial_crop.append([1, 1])

            # For smaller base sizes, resize; for larger, pad
            if self.base_size <= 640:
                resized_image = image.resize((self.base_size, self.base_size), Image.LANCZOS)
                images_list.append(self.image_transform(resized_image).to(self.dtype))
            else:
                global_view = ImageOps.pad(
                    image, (self.base_size, self.base_size),
                    color=tuple(int(x * 255) for x in self.image_transform.mean)
                )
                images_list.append(self.image_transform(global_view).to(self.dtype))

            num_queries = math.ceil((self.base_size // self.patch_size) / self.downsample_ratio)
            tokenized_image = ([self.image_token_id] * num_queries + [self.image_token_id]) * num_queries
            tokenized_image += [self.image_token_id]

        return images_list, images_crop_list, images_spatial_crop, tokenized_image, crop_ratio

    def process_single_sample(self, messages: List[Dict]) -> Dict[str, Any]:
            """
            Process a single conversation into model inputs.
            """

            # --- 1. Setup ---
            images = []
            for message in messages:
                if "images" in message and message["images"]:
                    for img_data in message["images"]:
                        if img_data is not None:
                            pil_image = self.deserialize_image(img_data)
                            images.append(pil_image)

            if not images:
                raise ValueError("No images found in sample. Please ensure all samples contain images.")

            tokenized_str = []
            images_seq_mask = []
            images_list, images_crop_list, images_spatial_crop = [], [], []

            prompt_token_count = -1 # Index to start training
            assistant_started = False
            image_idx = 0

            # Add BOS token at the very beginning
            tokenized_str.append(self.bos_id)
            images_seq_mask.append(False)

            for message in messages:
                role = message["role"]
                content = message["content"]

                # Check if this is the assistant's turn
                if role == "<|Assistant|>":
                    if not assistant_started:
                        # This is the split point. All tokens added *so far*
                        # are part of the prompt.
                        prompt_token_count = len(tokenized_str)
                        assistant_started = True

                    # Append the EOS token string to the *end* of assistant content
                    content = f"{content.strip()} {self.tokenizer.eos_token}"

                # Split this message's content by the image token
                text_splits = content.split('<image>')

                for i, text_sep in enumerate(text_splits):
                    # Tokenize the text part
                    tokenized_sep = text_encode(self.tokenizer, text_sep, bos=False, eos=False)
                    tokenized_str.extend(tokenized_sep)
                    images_seq_mask.extend([False] * len(tokenized_sep))

                    # If this text is followed by an <image> tag
                    if i < len(text_splits) - 1:
                        if image_idx >= len(images):
                            raise ValueError(
                                f"Data mismatch: Found '<image>' token but no corresponding image."
                            )

                        # Process the image
                        image = images[image_idx]
                        img_list, crop_list, spatial_crop, tok_img, _ = self.process_image(image)

                        images_list.extend(img_list)
                        images_crop_list.extend(crop_list)
                        images_spatial_crop.extend(spatial_crop)

                        # Add image placeholder tokens
                        tokenized_str.extend(tok_img)
                        images_seq_mask.extend([True] * len(tok_img))

                        image_idx += 1 # Move to the next image

            # --- 3. Validation and Final Prep ---
            if image_idx != len(images):
                raise ValueError(
                    f"Data mismatch: Found {len(images)} images but only {image_idx} '<image>' tokens were used."
                )

            # If we never found an assistant message, we're in a weird state
            # (e.g., user-only prompt). We mask everything.
            if not assistant_started:
                print("Warning: No assistant message found in sample. Masking all tokens.")
                prompt_token_count = len(tokenized_str)

            # Prepare image tensors
            images_ori = torch.stack(images_list, dim=0)
            images_spatial_crop_tensor = torch.tensor(images_spatial_crop, dtype=torch.long)

            if images_crop_list:
                images_crop = torch.stack(images_crop_list, dim=0)
            else:
                images_crop = torch.zeros((1, 3, self.base_size, self.base_size), dtype=self.dtype)

            return {
                "input_ids": torch.tensor(tokenized_str, dtype=torch.long),
                "images_seq_mask": torch.tensor(images_seq_mask, dtype=torch.bool),
                "images_ori": images_ori,
                "images_crop": images_crop,
                "images_spatial_crop": images_spatial_crop_tensor,
                "prompt_token_count": prompt_token_count, # This is now accurate
            }

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        """Collate batch of samples"""
        batch_data = []

        # Process each sample
        for feature in features:
            try:
                processed = self.process_single_sample(feature['messages'])
                batch_data.append(processed)
            except Exception as e:
                print(f"Error processing sample: {e}")
                continue

        if not batch_data:
            raise ValueError("No valid samples in batch")

        # Extract lists
        input_ids_list = [item['input_ids'] for item in batch_data]
        images_seq_mask_list = [item['images_seq_mask'] for item in batch_data]
        prompt_token_counts = [item['prompt_token_count'] for item in batch_data]

        # Pad sequences
        input_ids = pad_sequence(input_ids_list, batch_first=True, padding_value=self.tokenizer.pad_token_id)
        images_seq_mask = pad_sequence(images_seq_mask_list, batch_first=True, padding_value=False)

        # Create labels
        labels = input_ids.clone()

        # Mask padding tokens
        labels[labels == self.tokenizer.pad_token_id] = -100

        # Mask image tokens (model shouldn't predict these)
        labels[images_seq_mask] = -100

        # Mask user prompt tokens when train_on_responses_only=True (only train on assistant responses)
        if self.train_on_responses_only:
            for idx, prompt_count in enumerate(prompt_token_counts):
                if prompt_count > 0:
                    labels[idx, :prompt_count] = -100

        # Create attention mask
        attention_mask = (input_ids != self.tokenizer.pad_token_id).long()

        # Prepare images batch (list of tuples)
        images_batch = []
        for item in batch_data:
            images_batch.append((item['images_crop'], item['images_ori']))

        # Stack spatial crop info
        images_spatial_crop = torch.cat([item['images_spatial_crop'] for item in batch_data], dim=0)

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
            "images": images_batch,
            "images_seq_mask": images_seq_mask,
            "images_spatial_crop": images_spatial_crop,
        }

<a name="Train"></a>
### Train the model
We use our new `DeepSeekOCRDataCollator` which will help in our vision finetuning setup.

In [ ]:
from transformers import Trainer, TrainingArguments
from unsloth import is_bf16_supported
FastVisionModel.for_training(model) # Enable for training!
model.gradient_checkpointing_enable()

data_collator = DeepSeekOCRDataCollator(
    tokenizer=tokenizer,
    model = model,
    image_size=640,
    base_size=1024,
    crop_mode=True,
    train_on_responses_only=True,
)
trainer = Trainer(
    model = model,
    tokenizer = tokenizer,
    data_collator = data_collator,
    train_dataset = final_dataset,
    args = TrainingArguments(
        # --- 1. CẤU HÌNH VRAM (QUAN TRỌNG NHẤT CHO 8GB) ---
        per_device_train_batch_size = 1,  # [SỬA] Giảm xuống 1 để an toàn tuyệt đối cho VRAM
        gradient_accumulation_steps = 8,  # [SỬA] Tăng lên để bù cho Batch Size nhỏ (1x8 = hiệu quả tương đương batch 8)
        gradient_checkpointing = True,

        # --- 2. CẤU HÌNH HUẤN LUYỆN (LOGIC CHO 1372 MẪU) ---
        warmup_steps = 20,
        max_steps = 100,

        # --- 3. CÁC THAM SỐ KHÁC ---
        learning_rate = 2e-5,
        optim = "paged_adamw_8bit",
        save_steps=100,

        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        fp16 = not is_bf16_supported(),
        bf16 = is_bf16_supported(),
        output_dir = "output",
        report_to = "none",

        dataloader_num_workers = 0,
        dataloader_pin_memory = False,
        remove_unused_columns = False,
    ),
)

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA GeForce RTX 3070 Laptop GPU. Max memory = 8.0 GB.
6.543 GB of memory reserved.


In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3,984 | Num Epochs = 1 | Total steps = 10
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 77,509,632 of 3,413,615,872 (2.27% trained)
Unsloth: Not an error, but DeepseekOCRForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([9, 100, 1280])
Unsloth: Will smartly offload gradients to save VRAM!
BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([9, 100, 1280])
BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([9, 100, 1280])
BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([9, 100, 1280])
BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([9, 100, 1280])
BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([9, 100, 1280])
BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([9, 100, 1280])
BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([9, 100, 1280])


Step,Training Loss


BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([9, 100, 1280])
BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([9, 100, 1280])
BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([9, 100, 1280])
BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([9, 100, 1280])
BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([9, 100, 1280])
BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([9, 100, 1280])
BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([9, 100, 1280])
BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([8, 100, 1280])
BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([9, 100, 1280])
BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([9, 100, 1280])
BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([9, 100, 1280])
BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([9, 100, 1280])
BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([9, 100, 1280])
BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([9, 100, 1280])
BASE: 

You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR. This is not supported for all configurations of models and can yield errors.


In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

717.4787 seconds used for training.
11.96 minutes used for training.
Peak reserved memory = 16.246 GB.
Peak reserved memory for training = 9.703 GB.
Peak reserved memory % of max memory = 203.075 %.
Peak reserved memory for training % of max memory = 121.287 %.


<a name="Inference"></a>
### Inference
Let's run the model!

In [ ]:
prompt = "<image>\nConvert the following image of Vietnamese writing into digital text, keeping the tone diacritics and vowel diacritics in Vietnamese text."
image_file = 'UIT_HWDB_line/test_data/255/6.jpg'
output_path = 'output'

# Tiny: base_size = 512, image_size = 512, crop_mode = False
# Small: base_size = 640, image_size = 640, crop_mode = False
# Base: base_size = 1024, image_size = 1024, crop_mode = False
# Large: base_size = 1280, image_size = 1280, crop_mode = False

# Gundam: base_size = 1024, image_size = 640, crop_mode = True

res = model.infer(tokenizer, prompt=prompt, image_file=image_file,
    output_path = output_path,
    image_size=512,
    base_size=512,
    crop_mode=True,
    save_results = True,
    test_compress = False)


<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Huggingface's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [ ]:
model.save_pretrained("lora_model")  # Local saving
tokenizer.save_pretrained("lora_model")
# model.push_to_hub("your_name/lora_model", token = "...") # Online saving
# tokenizer.push_to_hub("your_name/lora_model", token = "...") # Online saving

OSError: [Errno 22] Invalid argument

# Validation

Evaluate Function

In [ ]:
import unicodedata
from jiwer import cer

In [ ]:
# Bỏ toàn bộ dấu tiếng Việt
def strip_diacritics(text: str) -> str:
    text = unicodedata.normalize("NFD", text)
    text = "".join(c for c in text if unicodedata.category(c) != "Mn")
    return unicodedata.normalize("NFC", text)

# Tách base char và dấu (để tính DER)
def extract_diacritics(text: str):
    base_chars = []
    diacritics = []
    for ch in text:
        decomp = unicodedata.normalize("NFD", ch)
        base = "".join(c for c in decomp if unicodedata.category(c) != "Mn")
        mark = "".join(c for c in decomp if unicodedata.category(c) == "Mn")
        base_chars.append(base)
        diacritics.append(mark)
    return "".join(base_chars), "".join(diacritics)

In [ ]:
def diacritic_error_rate(gt: str, pred: str) -> float:
    _, gt_diac = extract_diacritics(gt)
    _, pr_diac = extract_diacritics(pred)

    # CER trên chuỗi dấu
    if len(gt_diac) == 0:
        return 0.0
    return cer([gt_diac], [pr_diac])


In [ ]:
def evaluate_ocr_output(output_dir: str, gt_text: str, verbose=True):
    """
    Đánh giá OCR cho 1 ảnh:
    - CER (character error rate)
    - DER (diacritic error rate)
    - Qualitative error (in ra so sánh)
    """

    result_path = os.path.join(output_dir, "result.mmd")

    if not os.path.exists(result_path):
        raise FileNotFoundError(f"Không tìm thấy {result_path}")

    # Load prediction
    with open(result_path, encoding="utf-8") as f:
        pred_text = f.read().strip()

    gt_text = gt_text.strip()

    # === METRICS ===
    cer_score = cer([gt_text], [pred_text])
    der_score = diacritic_error_rate(gt_text, pred_text)

    # CER không dấu (phân tích phụ)
    cer_no_diac = cer(
        [strip_diacritics(gt_text)],
        [strip_diacritics(pred_text)]
    )

    results = {
        "CER": cer_score,
        "DER": der_score,
        "CER_no_diacritic": cer_no_diac,
        "prediction": pred_text,
        "ground_truth": gt_text,
    }

    # === QUALITATIVE ERROR ===
    if verbose:
        print("=== QUALITATIVE OCR EVALUATION ===")
        print("Ground Truth :", gt_text)
        print("Prediction   :", pred_text)
        print("--------------------------------")
        print(f"CER            : {cer_score:.4f}")
        print(f"DER (dấu)      : {der_score:.4f}")
        print(f"CER (no dấu)   : {cer_no_diac:.4f}")

    return results


Crawling

In [ ]:
TEST_ROOT = "data\\UIT_HWDB_line\\test_data"

test_raw_data = crawl_and_prepare_data(
    root_path=TEST_ROOT,
    seed=42,
    select_count=9999
)

print(f"Số mẫu test: {len(test_raw_data)}")


--> Chọn 6 thư mục từ data\UIT_HWDB_line\test_data
Số mẫu test: 201


In [ ]:
import random
random.seed(14)
sample = random.choice(test_raw_data)

img_path = sample["image_path"]
gt_text  = sample["text"].strip()

print("Image:", img_path)
print("GT   :", gt_text)


Image: data\UIT_HWDB_line\test_data\253\28.jpg
GT   : của người lao động - đồng bào - hay không?


In [ ]:
import gc
import torch

# Xoá tham chiếu
del model
del tokenizer

# Dọn Python memory
gc.collect()

# Dọn GPU memory
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

In [ ]:
model, tokenizer = FastVisionModel.from_pretrained(
    "deepseek_ocr",
    load_in_4bit = False, # Use 4bit to reduce memory use. False for 16bit LoRA.
    auto_model = AutoModel,
    trust_remote_code=True,
    unsloth_force_compile=True,
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for long context

)

You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR. This is not supported for all configurations of models and can yield errors.


Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2025.12.9: Fast Deepseekocr patching. Transformers: 4.56.2.
   \\   /|    NVIDIA GeForce RTX 3070 Laptop GPU. Num GPUs = 1. Max memory: 8.0 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.9.1+cu126. CUDA: 8.6. CUDA Toolkit: 12.6. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


'(ProtocolError('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')), '(Request ID: 8e8b01c3-8444-44d0-b73b-9bb209d757e6)')' thrown while requesting GET https://huggingface.co/unslothai/other/resolve/43d9e0f2f19a5d7836895f648dc0e762816acf77/model.safetensors
[huggingface_hub.utils._http|WARNING]'(ProtocolError('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')), '(Request ID: 8e8b01c3-8444-44d0-b73b-9bb209d757e6)')' thrown while requesting GET https://huggingface.co/unslothai/other/resolve/43d9e0f2f19a5d7836895f648dc0e762816acf77/model.safetensors
Retrying in 1s [Retry 1/5].
[huggingface_hub.utils._http|WARNING]Retrying in 1s [Retry 1/5].
You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR. This is not supported for all configurations of models and can yield errors.
You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR. This is not support

Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Some weights of DeepseekOCRForCausalLM were not initialized from the model checkpoint at deepseek_ocr and are newly initialized: ['model.vision_model.embeddings.position_ids']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
prompt = "<image>\nFree OCR."
output_path = "output"
os.makedirs(output_path, exist_ok=True)


model.eval()

with torch.no_grad():
    res = model.infer(
        tokenizer,
        prompt=prompt,
        image_file=img_path,
        output_path=output_path,
        base_size=1024,
        image_size=640,
        crop_mode=True,
        save_results=True,
        test_compress=False
    )

results = evaluate_ocr_output(
    output_dir="output",
    gt_text=gt_text,
    verbose=True
)


BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([7, 100, 1280])
ou a' ngu'di bòdìny - dòrì bòs' - hì yì hòmìy?
===============save results:===============


image: 0it [00:00, ?it/s]
other: 0it [00:00, ?it/s]

=== QUALITATIVE OCR EVALUATION ===
Ground Truth : của người lao động - đồng bào - hay không?
Prediction   : ou a' ngu'di bòdìny - dòrì bòs' - hì yì hòmìy?
--------------------------------
CER            : 0.6905
DER (dấu)      : 0.7000
CER (no dấu)   : 0.5714


### Fine-tuned model

In [ ]:
import gc
import torch

# Xoá tham chiếu
del model
del tokenizer

# Dọn Python memory
gc.collect()

# Dọn GPU memory
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

NameError: name 'model' is not defined

In [ ]:
if True:
    model, tokenizer = FastVisionModel.from_pretrained(
        model_name = "output/checkpoint-10", # YOUR MODEL YOU USED FOR TRAINING
        load_in_4bit = False, # Use 4bit to reduce memory use. False for 16bit LoRA.
        auto_model = AutoModel,
        trust_remote_code=True,
        unsloth_force_compile=True,
        use_gradient_checkpointing = "unsloth", # True or "unsloth" for long context
    )
    FastVisionModel.for_inference(model) # Enable for inference!

print(f"Model is on: {model.device}")

You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR. This is not supported for all configurations of models and can yield errors.
You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR. This is not supported for all configurations of models and can yield errors.


Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2025.12.9: Fast Deepseekocr patching. Transformers: 4.56.2.
   \\   /|    NVIDIA GeForce RTX 3070 Laptop GPU. Num GPUs = 1. Max memory: 8.0 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.9.1+cu126. CUDA: 8.6. CUDA Toolkit: 12.6. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR. This is not supported for all configurations of models and can yield errors.
You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR. This is not supported for all configurations of models and can yield errors.


Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Some weights of DeepseekOCRForCausalLM were not initialized from the model checkpoint at ./deepseek_ocr and are newly initialized: ['model.vision_model.embeddings.position_ids']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model is on: cuda:0


In [ ]:
output_path = "output"
os.makedirs(output_path, exist_ok=True)
model.eval()

with torch.no_grad():
    res = model.infer(
        tokenizer,
        prompt=INSTRUCTION,
        image_file=img_path,
        output_path=output_path,
        base_size=1024,
        image_size=640,
        crop_mode=True,
        save_results=True,
        test_compress=False
    )

results = evaluate_ocr_output(
    output_dir="output",
    gt_text=gt_text,
    verbose=True
)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([7, 100, 1280])

===============save results:===============


image: 0it [00:00, ?it/s]
other: 0it [00:00, ?it/s]

=== QUALITATIVE OCR EVALUATION ===
Ground Truth : của người lao động - đồng bào - hay không?
Prediction   : 
--------------------------------
CER            : 1.0000
DER (dấu)      : 1.0000
CER (no dấu)   : 1.0000
